In [17]:
from data import *

from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [18]:
# generate data
d1 = 1000
d2 = 1000
r = 5
p = 5 / d2

M = get_random_matrix(d1, d2, r) / np.sqrt(d1)
O = np.random.rand(d1, d2) < 0.04
M = M*O
M_masks = (M>0) * 1
#print(M)
observed_M, masks = get_uniformly_random_samples(M, p)
#print(d1*d2)
#print(p)
#print(masks.sum())
#print(observed_M)

Non-zero elements do not align correctly with the mask.


In [19]:
#eps = 0.01
MTM = M.T @ M
cov_observe_M =  observed_M.T @ observed_M
cov_observe_count = (observed_M == 0).T @ (observed_M == 0)
diag_cov = np.diag( np.diag(cov_observe_M) )
print(cov_observe_count)
np.count_nonzero(cov_observe_M)

[[ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 ...
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]
 [ True  True  True ...  True  True  True]]


220

In [20]:
num_entries = (M>0).sum()
ratio = num_entries/(d1*d2)
print(num_entries/(d1*d2))

0.040069


In [21]:
cov_observe_count = (1 * (observed_M != 0)).T @ (1 * (observed_M != 0))
cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
MTM_count = (1 * (M != 0)).T @ (1 * (M != 0))
MTM_count = MTM_count + (MTM_count == 0) * 1
#print(cov_observe_count)
#T = cov_observe_M / (cov_observe_count/d1)
T = cov_observe_M / (cov_observe_count/(d1))
MT = MTM / (MTM_count/(d1))
print(T)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [22]:
T_masks = np.nonzero(T)

mask_err = T[T_masks] - MT[T_masks]
#print(mask_err)
print(np.linalg.norm(mask_err) / np.linalg.norm(MT, 'fro'))

0.0019271491549098048


In [23]:
T_p = (1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov)

mask_err_Tp = T_p[T_masks] - MTM[T_masks]

print(np.linalg.norm(mask_err_Tp) / np.linalg.norm(MTM, 'fro'))

104.74226512891833


In [24]:
# impute missing values from rank-r SVD corresponding to masks

train_losses = []
err_estimates = []

epochs = 100
#lr = 0.05
X = T
T_masks = 1 * (T != 0)
loop = tqdm(range(epochs))
for i in loop:
    U, D, Vt = np.linalg.svd(X)
    D[r:] = 0
    X_update = U @ np.diag(D) @ Vt

    X = T * T_masks + X_update * (1 - T_masks)

    err = MTM - X
    loss = (err**2).mean()
    train_losses.append(loss)
    relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(MTM, 'fro') 
    err_estimates.append(relative_err)
    loop.set_description(f"relative err: {relative_err:.7f}")
    #print(relative_err)

plt.figure(figsize=(5, 3))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot(err_estimates, label='Err Estimation')
plt.xlabel('Epoch')
plt.ylabel('Err')
plt.title('Err Estimation')
plt.legend()

relative err: 6.0705041:  44%|████▍     | 44/100 [00:13<00:16,  3.35it/s]


KeyboardInterrupt: 

In [43]:
# run soft impute
epochs = 30
lr = 0.05
X = T
for i in range(epochs):
    U, D, Vt = np.linalg.svd(X)
    D[r:] = 0
    X_update = U @ np.diag(D) @ Vt

    #if np.linalg.norm(X - X_update, 'fro') / np.linalg.norm(X) < eps:
    #    print(i)
    #    break

    X = (1 - lr) * X + lr * X_update

    # print distance
    err = M.T @ M - X
    relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M)
    print(relative_err)

#print(D)

0.9971418921930277
0.9969828642269022
0.9968488577461831
0.9967369635820517
0.9966445781544958
0.9965693729174921
0.9965092667514615
0.9964624010311309
0.9964271171199677
0.9964019360638401
0.9963855402764987
0.9963767570279076
0.9963745435634674
0.9963779736978146
0.9963862257412324
0.996398571629857
0.9964143671428799
0.9964330431009168
0.9964540974497093
0.996477088142427
0.9965016267421203
0.9965273726733823
0.9965540280591144
0.996581333084474
0.9966090618356933
0.9966370185665476
0.9966650343498389
0.9966929640754415
0.9967206837602043
0.9967480881384212


In [44]:
err = M.T @ M - T
relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M)
print(relative_err)

0.9973291902007744


In [45]:
cov_observe_count

array([[3, 1, 1, ..., 1, 1, 1],
       [1, 3, 1, ..., 1, 1, 1],
       [1, 1, 5, ..., 1, 1, 1],
       ...,
       [1, 1, 1, ..., 4, 1, 1],
       [1, 1, 1, ..., 1, 4, 1],
       [1, 1, 1, ..., 1, 1, 4]])

In [46]:
T

array([[1.45743539, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.5332217 , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.8411337 , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.73838423, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.19088458,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.73258031]])